# Imports

In [1]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product
from scipy.stats import norm
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Varying seeds

## load data

In [16]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/diff_seed'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'seq_len', 'pred_len', 'data_id', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'batch_size', 'auxi_type', 'auxi_mode', 'lradj', 'patience', 'train_epochs', 'fix_seed']
metric_names = ['mse', 'mae']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    result.loc[:, metric_names] = metric[1], metric[0]
    result.loc[:, ['exp_dir']] = exp_dir
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

## preprocess

In [54]:
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'

baselines = pd.read_csv(f'{save_root}/baselines_params.csv')
finetunes_best = pd.read_csv(f'{save_root}/best_finetune_full_each.csv')

base = baselines.copy()
base = base[
    (base.model == 'iTransformer') &
    (base.data_id.isin(['ECL', 'Weather']))
]
base['label'] = 'DF'
base['fix_seed'] = 2023


best = finetunes_best.copy()
best = best[
    (best.model == 'iTransformer') &
    (best.data_id.isin(['ECL_PCA', 'Weather_PCA']))
]
best = best[best.pred_len != 'Avg']

best['fix_seed'] = 2023
best['data_id'] = best['data_id'].str.replace('_PCA', '')
best['label'] = 'PDF'

df2 = df.copy()
df2 = df2[
    (df2.model == 'iTransformer') &
    (df2.data_id.isin(['ECL_PCA', 'Weather_PCA', 'ECL', 'Weather']))
]
df2['label'] = df2['data_id'].apply(lambda x: 'PDF' if '_PCA' in x else 'DF')
df2['data_id'] = df2['data_id'].str.replace('_PCA', '')


columns = ['pred_len', 'data_id', 'fix_seed', 'mse', 'mae', 'label']
df_seed = pd.concat([base[columns], best[columns], df2[columns]], ignore_index=True)
df_seed['pred_len'] = df_seed['pred_len'].astype(str)
df_seed['fix_seed'] = df_seed['fix_seed'].astype(int)

dst_order = ['ECL', 'Weather']
df_seed['data_id'] = pd.Categorical(df_seed['data_id'], categories=dst_order, ordered=True)

seed_order = [2021, 2022, 2023, 2024, 2025]
df_seed['fix_seed'] = pd.Categorical(df_seed['fix_seed'], categories=seed_order, ordered=True)

pred_len_order = ['96', '192', '336', '720', 'Avg']
df_seed['pred_len'] = pd.Categorical(df_seed['pred_len'], categories=pred_len_order, ordered=True)

df_seed_avg = df_seed.groupby(['data_id', 'fix_seed', 'label']).mean(numeric_only=True).reset_index()
df_seed_avg['pred_len'] = 'Avg'

df_seed = pd.concat([df_seed, df_seed_avg], ignore_index=True)

df_seed.sort_values(by=['data_id', 'fix_seed', 'label', 'pred_len'], inplace=True)


df_seed_mean_seed = df_seed.groupby(['data_id', 'label', 'pred_len']).mean(numeric_only=True).reset_index()
df_seed_std_seed = df_seed.groupby(['data_id', 'label', 'pred_len']).std(numeric_only=True).reset_index()

df_seed_agg = pd.merge(df_seed_mean_seed, df_seed_std_seed, on=['data_id', 'label', 'pred_len'], suffixes=('_mean', '_std'))
df_seed_agg['mse'] = df_seed_agg['mse_mean'].apply(lambda x: "{:.3f}".format(x)) + '$_{\pm ' + df_seed_agg['mse_std'].apply(lambda x: "{:.3f}".format(x)) + '}$'
df_seed_agg['mae'] = df_seed_agg['mae_mean'].apply(lambda x: "{:.3f}".format(x)) + '$_{\pm ' + df_seed_agg['mae_std'].apply(lambda x: "{:.3f}".format(x)) + '}$'
df_seed_agg = df_seed_agg[['data_id', 'label', 'pred_len', 'mse', 'mae']]

df_seed_agg['pred_len'] = pd.Categorical(df_seed_agg['pred_len'], categories=pred_len_order, ordered=True)
df_seed_agg.sort_values(by=['data_id', 'label', 'pred_len'], inplace=True)

/tmp/ipykernel_445196/2341988222.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_seed_avg = df_seed.groupby(['data_id', 'fix_seed', 'label']).mean(numeric_only=True).reset_index()
/tmp/ipykernel_445196/2341988222.py:57: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_seed_mean_seed = df_seed.groupby(['data_id', 'label', 'pred_len']).mean(numeric_only=True).reset_index()
/tmp/ipykernel_445196/2341988222.py:58: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the

In [56]:
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
df_seed_agg.to_csv(f'{save_root}/seed_stats.csv', index=False)

df_seed_agg

,data_id,label,pred_len,mse,mae
3,ECL,DF,96,0.150$_{\pm 0.001}$,0.242$_{\pm 0.001}$
0,ECL,DF,192,0.166$_{\pm 0.002}$,0.257$_{\pm 0.002}$
1,ECL,DF,336,0.181$_{\pm 0.001}$,0.273$_{\pm 0.001}$
2,ECL,DF,720,0.216$_{\pm 0.004}$,0.303$_{\pm 0.003}$
4,ECL,DF,Avg,0.178$_{\pm 0.001}$,0.269$_{\pm 0.001}$
8,ECL,PDF,96,0.145$_{\pm 0.000}$,0.235$_{\pm 0.000}$
5,ECL,PDF,192,0.160$_{\pm 0.001}$,0.249$_{\pm 0.001}$
6,ECL,PDF,336,0.174$_{\pm 0.002}$,0.266$_{\pm 0.002}$
7,ECL,PDF,720,0.205$_{\pm 0.001}$,0.293$_{\pm 0.001}$
9,ECL,PDF,Avg,0.171$_{\pm 0.001}$,0.261$_{\pm 0.001}$
